In [2]:
"""
AzCon Unified Feature Store + Segmentation + Recommendation Engine
==================================================================
5 şirkət: Aztelekom, AZAL, Azərpoçt, ADY, Azİntelekom
- Hər şirkətin DB-si ayrıca CSV kimi saxlanılır
- Feature Store birləşdirilir + engineered features
- RandomForest → churn_score
- KMeans → müştəri seqmentləri
- Behaviour-based cross-company recommendation engine
"""

import numpy as np
import pandas as pd
from functools import reduce
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ─────────────────────────────────────────────
# 0. OUTPUT DIRECTORY
# ─────────────────────────────────────────────
OUT = "azcon_data"
os.makedirs(OUT, exist_ok=True)

# ─────────────────────────────────────────────
# 1. GLOBAL KEYS
# ─────────────────────────────────────────────
N = 2000
rng = np.random.default_rng(42)

prefixes = rng.choice(["050", "051", "055", "070", "077"], size=N)
suffixes = rng.integers(1_000_000, 9_999_999, size=N)
phone_numbers = np.array([f"+994{p}{s}" for p, s in zip(prefixes, suffixes)])

# Latent indices (used across all builders)
wealth_index  = rng.beta(2, 3, size=N)
digital_index = rng.beta(2, 2, size=N)

global_keys = pd.DataFrame({"phone_number": phone_numbers})


# ─────────────────────────────────────────────
# 2. DATABASE BUILDERS
# ─────────────────────────────────────────────

def build_aztelekom(phone_numbers, wealth_index, digital_index, rng):
    n = len(phone_numbers)

    monthly_spend      = (10 + wealth_index * 180 + rng.normal(0, 10, n)).clip(5, 250)
    tenure_months      = (rng.integers(1, 12, n) + wealth_index * 48).astype(int).clip(1, 120)
    data_usage_gb      = (digital_index * 300 + rng.normal(0, 40, n)).clip(10, 600)
    speed_mbps         = rng.choice([20, 50, 100, 200], n)
    support_calls      = rng.poisson(1.5, n)
    complaints         = rng.poisson(1.2, n)
    downtime_hours     = rng.exponential(2, n).clip(0, 20)
    payment_delay_days = rng.poisson(1, n)
    device_count       = rng.integers(1, 6, n)
    plan_upgrade_count = rng.poisson(1, n)
    contract_type      = rng.choice([0, 1], n)           # 0=prepaid, 1=postpaid
    region             = rng.integers(1, 5, n)
    payment_status     = rng.choice(["paid", "unpaid", "overdue"], n, p=[0.75, 0.15, 0.10])
    service_type       = rng.choice(["GPON", "ADSL", "Cable", "5G"], n)
    average_monthly_usage_gb = data_usage_gb + rng.normal(0, 20, n).clip(-30, 30)
    peak_usage_time    = rng.choice(["morning", "afternoon", "evening", "night"], n,
                                     p=[0.15, 0.20, 0.45, 0.20])
    installation_address_airport_km = rng.exponential(15, n).clip(1, 80)
    customer_satisfaction_score = rng.integers(1, 6, n)
    smart_home_devices = rng.poisson(0.8, n)
    azcloud_subscriber = (digital_index > 0.65).astype(int)
    marketing_consent  = (rng.random(n) < 0.72).astype(int)

    churn_prob   = (support_calls * 0.08 + downtime_hours * 0.05).clip(0, 1)
    churn_status = (rng.random(n) < churn_prob).astype(int)
    cltv         = monthly_spend * tenure_months

    return pd.DataFrame({
        "phone_number": phone_numbers,
        # Original features
        "monthly_spend": monthly_spend.round(2),
        "data_usage_gb": data_usage_gb.round(1),
        "speed_mbps": speed_mbps,
        "support_calls": support_calls,
        "complaints": complaints,
        "downtime_hours": downtime_hours.round(2),
        "payment_delay_days": payment_delay_days,
        "device_count": device_count,
        "plan_upgrade_count": plan_upgrade_count,
        "contract_type": contract_type,
        "region": region,
        "tenure_months": tenure_months,
        "churn_status": churn_status,
        "cltv": cltv.round(2),
        # New features (recommendation üçün)
        "payment_status": payment_status,
        "service_type": service_type,
        "average_monthly_usage_gb": average_monthly_usage_gb.round(1),
        "peak_usage_time": peak_usage_time,
        "installation_address_airport_km": installation_address_airport_km.round(1),
        "customer_satisfaction_score": customer_satisfaction_score,
        "smart_home_devices": smart_home_devices,
        "azcloud_subscriber": azcloud_subscriber,
        "marketing_consent": marketing_consent,
    })


def build_azal(phone_numbers, wealth_index, rng):
    n = len(phone_numbers)

    is_business = rng.random(n) < (0.1 + wealth_index * 0.5)

    flights_per_month  = rng.poisson(1.5, n)
    total_spent        = np.where(is_business,
                                   rng.normal(1200, 400, n),
                                   rng.normal(200, 100, n)).clip(50, 3000)
    last_flight_days   = rng.integers(1, 365, n)
    intl_ratio         = rng.random(n)
    business_ratio     = np.where(is_business,
                                   rng.uniform(0.5, 1, n),
                                   rng.uniform(0, 0.3, n))
    delays             = rng.poisson(1, n)
    cancellations      = rng.poisson(0.3, n)
    loyalty_points     = rng.integers(0, 50000, n)
    avg_ticket_price   = total_spent / (flights_per_month + 1)
    baggage_score      = rng.integers(1, 6, n)
    checkin_score      = rng.integers(1, 6, n)
    satisfaction       = np.where((baggage_score + checkin_score) < 5, 0, 1)
    # New features
    travel_type        = np.where(is_business, "Business", "Leisure")
    extra_baggage_tendency = rng.integers(0, 4, n)       # average extra bags per flight
    next_booked_flight_days = rng.integers(-7, 90, n)    # negative = already past (no booking)
    refund_requests    = rng.poisson(0.3, n)
    lounge_visits      = rng.poisson(0.5, n) * is_business.astype(int)
    domestic_flight_ratio = 1 - intl_ratio
    marketing_consent  = (rng.random(n) < 0.68).astype(int)

    return pd.DataFrame({
        "phone_number": phone_numbers,
        "flights_per_month": flights_per_month,
        "total_spent": total_spent.round(2),
        "avg_ticket_price": avg_ticket_price.round(2),
        "last_flight_days": last_flight_days,
        "intl_ratio": intl_ratio.round(3),
        "business_ratio": business_ratio.round(3),
        "delays": delays,
        "cancellations": cancellations,
        "loyalty_points": loyalty_points,
        "baggage_score": baggage_score,
        "checkin_score": checkin_score,
        "satisfaction": satisfaction,
        # New
        "travel_type": travel_type,
        "extra_baggage_tendency": extra_baggage_tendency,
        "next_booked_flight_days": next_booked_flight_days,
        "refund_requests": refund_requests,
        "lounge_visits": lounge_visits,
        "domestic_flight_ratio": domestic_flight_ratio.round(3),
        "marketing_consent": marketing_consent,
    })


def build_azerpoct(phone_numbers, wealth_index, digital_index, rng):
    n = len(phone_numbers)

    parcels_sent     = rng.poisson(5, n)
    parcels_received = rng.poisson(5, n)
    avg_parcel_value = (wealth_index * 200 + rng.normal(20, 10, n)).clip(5, 500)
    intl_shipments   = rng.poisson(1, n)
    delivery_delay   = rng.poisson(1, n)
    lost_parcels     = rng.poisson(0.1, n)
    pickup_usage     = rng.choice([0, 1], n)
    home_delivery_ratio = rng.random(n)
    business_client  = (parcels_sent > 7).astype(int)
    visa_card        = (digital_index > 0.6).astype(int)
    # New features
    preferred_delivery_method = np.where(home_delivery_ratio > 0.6, "home", "branch")
    money_transfers_frequency = rng.poisson(0.8, n)
    ecommerce_shipper         = (parcels_sent > 10).astype(int)
    utility_payments_at_branch = rng.integers(0, 5, n)
    international_destination  = rng.choice(
        ["Russia", "Turkey", "UAE", "UK", "Germany", "None"], n,
        p=[0.20, 0.25, 0.20, 0.10, 0.05, 0.20])
    shipment_cancelled_lastmin = rng.poisson(0.2, n)
    marketing_consent          = (rng.random(n) < 0.65).astype(int)

    return pd.DataFrame({
        "phone_number": phone_numbers,
        "parcels_sent": parcels_sent,
        "parcels_received": parcels_received,
        "avg_parcel_value": avg_parcel_value.round(2),
        "intl_shipments": intl_shipments,
        "delivery_delay": delivery_delay,
        "lost_parcels": lost_parcels,
        "pickup_usage": pickup_usage,
        "home_delivery_ratio": home_delivery_ratio.round(3),
        "business_client": business_client,
        "visa_card": visa_card,
        # New
        "preferred_delivery_method": preferred_delivery_method,
        "money_transfers_frequency": money_transfers_frequency,
        "ecommerce_shipper": ecommerce_shipper,
        "utility_payments_at_branch": utility_payments_at_branch,
        "international_destination": international_destination,
        "shipment_cancelled_lastmin": shipment_cancelled_lastmin,
        "marketing_consent": marketing_consent,
    })


def build_ady(phone_numbers, digital_index, rng):
    n = len(phone_numbers)

    trips          = rng.poisson(2, n)
    spend          = rng.normal(60, 30, n).clip(10, 300)
    app_ratio      = digital_index
    cash_ratio     = 1 - app_ratio
    ticket_class   = np.where(app_ratio > 0.6, "Business", "Standard")
    loyalty        = trips * rng.integers(50, 200, n)
    delay          = rng.poisson(1, n)
    cancellations  = rng.poisson(0.2, n)
    group_travel   = rng.choice([0, 1], n, p=[0.7, 0.3])
    # New features
    route_type     = rng.choice(["domestic", "nakhchivan", "international"], n,
                                 p=[0.55, 0.25, 0.20])
    seasonal_traveler = rng.choice([0, 1], n, p=[0.6, 0.4])
    marketing_consent = (rng.random(n) < 0.70).astype(int)
    heavy_luggage  = rng.choice([0, 1], n, p=[0.65, 0.35])

    return pd.DataFrame({
        "phone_number": phone_numbers,
        "trips": trips,
        "spend": spend.round(2),
        "app_ratio": app_ratio.round(3),
        "cash_ratio": cash_ratio.round(3),
        "ticket_class": ticket_class,
        "loyalty": loyalty,
        "delay": delay,
        "cancellations": cancellations,
        "group_travel": group_travel,
        # New
        "route_type": route_type,
        "seasonal_traveler": seasonal_traveler,
        "heavy_luggage": heavy_luggage,
        "marketing_consent": marketing_consent,
    })


def build_azintelekom(phone_numbers, digital_index, rng):
    """Azİntelekom — rəqəmsal kimlik platforması"""
    n = len(phone_numbers)

    verified       = (rng.random(n) < (0.3 + digital_index * 0.6)).astype(int)
    login_freq     = (digital_index * 40 + rng.normal(5, 3, n)).clip(0, 60)
    services       = (digital_index * 10).astype(int)
    security_alerts = rng.poisson(0.5, n)
    backup_usage   = rng.choice([0, 1], n)
    # New features
    digital_literacy_score = rng.choice(["Low", "Medium", "High"], n,
                                         p=[0.2, 0.45, 0.35])
    primary_document_types = rng.choice(["Personal", "Business", "Banking", "Government"], n,
                                         p=[0.40, 0.25, 0.20, 0.15])
    biometric_enabled      = (digital_index > 0.55).astype(int)
    sso_usage              = (digital_index > 0.70).astype(int)
    data_sync_frequency    = rng.choice(["Never", "Weekly", "Daily"], n,
                                         p=[0.20, 0.35, 0.45])
    govt_services_usage    = rng.integers(0, 10, n)
    digital_signature_used = rng.poisson(1.2, n)
    marketing_consent      = (rng.random(n) < 0.75).astype(int)

    return pd.DataFrame({
        "phone_number": phone_numbers,
        "verified": verified,
        "login_freq": login_freq.round(1),
        "services": services,
        "security_alerts": security_alerts,
        "backup_usage": backup_usage,
        # New
        "digital_literacy_score": digital_literacy_score,
        "primary_document_types": primary_document_types,
        "biometric_enabled": biometric_enabled,
        "sso_usage": sso_usage,
        "data_sync_frequency": data_sync_frequency,
        "govt_services_usage": govt_services_usage,
        "digital_signature_used": digital_signature_used,
        "marketing_consent": marketing_consent,
    })


# ─────────────────────────────────────────────
# 3. BUILD ALL DATABASES
# ─────────────────────────────────────────────
print("=" * 60)
print("AzCon Feature Store Pipeline")
print("=" * 60)

aztelekom_df = build_aztelekom(phone_numbers, wealth_index, digital_index, rng)
azal_df      = build_azal(phone_numbers, wealth_index, rng)
azerpoct_df  = build_azerpoct(phone_numbers, wealth_index, digital_index, rng)
ady_df       = build_ady(phone_numbers, digital_index, rng)
azintelekom_df = build_azintelekom(phone_numbers, digital_index, rng)

# ─── Save individual DBs as CSV ───────────────
aztelekom_df.to_csv(f"{OUT}/aztelekom.csv", index=False)
azal_df.to_csv(f"{OUT}/azal.csv", index=False)
azerpoct_df.to_csv(f"{OUT}/azerpoct.csv", index=False)
ady_df.to_csv(f"{OUT}/ady.csv", index=False)
azintelekom_df.to_csv(f"{OUT}/azintelekom.csv", index=False)

print(f"\n✅ 5 şirkətin DB-si {OUT}/ qovluğuna saxlanıldı:")
for db, df in [("aztelekom", aztelekom_df), ("azal", azal_df),
               ("azerpoct", azerpoct_df), ("ady", ady_df), ("azintelekom", azintelekom_df)]:
    print(f"   {db}.csv — {df.shape[0]} sətir × {df.shape[1]} sütun")


# ─────────────────────────────────────────────
# 4. FEATURE STORE (JOIN)
# ─────────────────────────────────────────────

# marketing_consent hər df-də var — merge-dən ƏVVƏL çıxarırıq,
# sonra global_marketing_consent kimi bir dəfə əlavə edirik.
global_marketing_consent = (
    aztelekom_df["marketing_consent"].values |
    azal_df["marketing_consent"].values |
    azerpoct_df["marketing_consent"].values |
    ady_df["marketing_consent"].values |
    azintelekom_df["marketing_consent"].values
)

def drop_mc(df):
    return df.drop(columns=["marketing_consent"], errors="ignore")

feature_tables = [
    global_keys,
    drop_mc(aztelekom_df),
    drop_mc(azal_df),
    drop_mc(azerpoct_df),
    drop_mc(ady_df),
    drop_mc(azintelekom_df),
]

feature_store = reduce(
    lambda left, right: pd.merge(left, right, on="phone_number", how="left"),
    feature_tables
)

# Global consent flag əlavə et
feature_store["global_marketing_consent"] = global_marketing_consent

# ─────────────────────────────────────────────
# 5. ENGINEERED FEATURES
# ─────────────────────────────────────────────

# Value
feature_store["revenue_score"] = (
    feature_store["monthly_spend"] +
    feature_store["total_spent"] +
    feature_store["avg_parcel_value"] * 0.5
)

# Risk
feature_store["risk_score"] = (
    feature_store["support_calls"] +
    feature_store["downtime_hours"] +
    feature_store["delays"] +
    feature_store["delivery_delay"]
)

# Digital engagement
feature_store["digital_score"] = (
    feature_store["login_freq"] +
    feature_store["app_ratio"] * 20 +
    feature_store["verified"] * 8 +
    feature_store["biometric_enabled"] * 5 +
    feature_store["sso_usage"] * 3
)

# Loyalty composite
feature_store["loyalty_score"] = (
    feature_store["loyalty"] +
    feature_store["loyalty_points"] +
    feature_store["cltv"] / 100
)

# Mobility score (how often they travel / move)
feature_store["mobility_score"] = (
    feature_store["flights_per_month"] * 3 +
    feature_store["trips"] +
    feature_store["intl_ratio"] * 5
)

# Cross-sell potential (high value + multiple company interactions)
company_activity = (
    (feature_store["monthly_spend"] > 0).astype(int) +
    (feature_store["flights_per_month"] > 0).astype(int) +
    (feature_store["parcels_sent"] > 0).astype(int) +
    (feature_store["trips"] > 0).astype(int) +
    (feature_store["verified"] > 0).astype(int)
)
feature_store["cross_sell_potential"] = (
    company_activity * feature_store["revenue_score"] / 100
)

# Travel imminent flag
feature_store["travel_imminent"] = (
    (feature_store["next_booked_flight_days"] >= 0) &
    (feature_store["next_booked_flight_days"] <= 14)
).astype(int)

# High data user
feature_store["high_data_user"] = (feature_store["data_usage_gb"] > 350).astype(int)

# Business persona (multiple signals)
feature_store["is_business_persona"] = (
    ((feature_store["business_ratio"] > 0.5) |
     (feature_store["primary_document_types"] == "Business") |
     (feature_store["business_client"] == 1) |
     (feature_store["ticket_class"] == "Business")).astype(int)
)

feature_store.to_csv(f"{OUT}/feature_store.csv", index=False)
print(f"\n✅ feature_store.csv saxlanıldı — {feature_store.shape[0]} sətir × {feature_store.shape[1]} sütun")
print(f"   Null dəyərlər: {feature_store.isnull().sum().sum()}")


# ─────────────────────────────────────────────
# 6. CURATED FEATURES (numeric only, for ML)
# ─────────────────────────────────────────────
numeric_features = [
    # Value
    "monthly_spend", "total_spent", "avg_parcel_value", "cltv",
    # Risk
    "support_calls", "downtime_hours", "payment_delay_days", "complaints",
    # Travel & mobility
    "flights_per_month", "trips", "intl_ratio", "extra_baggage_tendency",
    "next_booked_flight_days", "last_flight_days",
    # Digital
    "login_freq", "verified", "app_ratio", "digital_literacy_score_enc",
    "biometric_enabled", "sso_usage",
    # Loyalty
    "loyalty_points", "loyalty",
    # Post/parcel
    "parcels_sent", "intl_shipments", "money_transfers_frequency",
    # Telco
    "data_usage_gb", "speed_mbps", "device_count", "tenure_months",
    "customer_satisfaction_score", "smart_home_devices",
    # Azİntelekom
    "services", "security_alerts", "digital_signature_used", "govt_services_usage",
    # Engineered
    "revenue_score", "risk_score", "digital_score", "loyalty_score",
    "mobility_score", "cross_sell_potential",
    # Flags
    "travel_imminent", "high_data_user", "is_business_persona",
]

# Encode ordinal categoricals
feature_store["digital_literacy_score_enc"] = feature_store["digital_literacy_score"].map(
    {"Low": 0, "Medium": 1, "High": 2}
)

curated = feature_store[["phone_number"] + numeric_features].copy()
curated = curated.fillna(0)  # Any join NULLs → 0

print(f"\n✅ Curated features: {curated.shape[1] - 1} feature, {curated.shape[0]} müştəri")


# ─────────────────────────────────────────────
# 7. CHURN MODEL (RandomForest)
# ─────────────────────────────────────────────
print("\n" + "─" * 40)
print("7. Churn Prediction Model")
print("─" * 40)

target   = "churn_status"
X        = curated[numeric_features]
y        = feature_store[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(f"ROC-AUC : {roc_auc_score(y_test, y_prob):.4f}")
print(classification_report(y_test, y_pred))

curated["churn_score"] = rf.predict_proba(X)[:, 1]
feature_store["churn_score"] = curated["churn_score"]


# ─────────────────────────────────────────────
# 8. CUSTOMER SEGMENTATION (KMeans)
# ─────────────────────────────────────────────
print("─" * 40)
print("8. Customer Segmentation (KMeans)")
print("─" * 40)

seg_features = [
    "revenue_score", "loyalty_score", "digital_score",
    "mobility_score", "risk_score", "cross_sell_potential",
    "is_business_persona", "churn_score"
]

scaler   = StandardScaler()
X_seg    = scaler.fit_transform(curated[seg_features])
kmeans   = KMeans(n_clusters=5, random_state=42, n_init=20)
clusters = kmeans.fit_predict(X_seg)
curated["cluster"] = clusters
feature_store["cluster"] = clusters

# ─── Name clusters based on centroid profiles ───
cluster_means = curated.groupby("cluster")[seg_features].mean()

def name_cluster(row):
    if row["churn_score"] > 0.55:
        return "High Risk"
    elif row["is_business_persona"] > 0.6 and row["mobility_score"] > row["mobility_score"]:
        return "Business Traveler"
    elif row["digital_score"] > cluster_means["digital_score"].mean() * 1.2:
        return "Digital Power User"
    elif row["loyalty_score"] > cluster_means["loyalty_score"].mean() * 1.2:
        return "Loyal VIP"
    else:
        return "Standard"

cluster_labels = cluster_means.apply(name_cluster, axis=1)
curated["segment"] = curated["cluster"].map(cluster_labels)
feature_store["segment"] = curated["segment"]

# Churn risk tier
def churn_tier(score):
    if score > 0.65:   return "🔴 High Risk"
    elif score > 0.40: return "🟡 Medium Risk"
    else:              return "🟢 Loyal"

curated["churn_tier"]       = curated["churn_score"].apply(churn_tier)
feature_store["churn_tier"] = curated["churn_tier"]

print("\nSegment distribution:")
print(curated["segment"].value_counts())
print("\nChurn tier distribution:")
print(curated["churn_tier"].value_counts())


# ─────────────────────────────────────────────
# 9. RECOMMENDATION ENGINE
# ─────────────────────────────────────────────

def azal_recs(row):
    recs = []

    # Səyahət yaxınlaşanda Aztelekom dondurmaq
    if row.get("travel_imminent", 0) == 1:
        recs.append({
            "company": "Aztelekom",
            "offer": "Ev interneti dondurma xidməti – Səyahət müddətinə pulsuz dondur",
            "trigger": "AZAL biletiniz var, evdə internet boş qalmasın",
            "biznes_deyeri": "Aztelekom: loyalty artımı; AZAL: müştəri bağlılığı"
        })
        # Airport uzaq isə transfer
        if row.get("installation_address_airport_km", 0) > 25:
            recs.append({
                "company": "ADY/Taksi",
                "offer": "Hava limanına xüsusi transfer xidməti",
                "trigger": "Eviniz hava limanına 25+ km uzaqdır",
                "biznes_deyeri": "AZAL: add-on gəlir; ADY: yeni müştəri"
            })

    # Biznes sərnişin → Aztelekom Dedicated VIP internet
    if row.get("travel_type", "") == "Business" or row.get("business_ratio", 0) > 0.5:
        recs.append({
            "company": "Aztelekom",
            "offer": "Dedicated VIP Ev İnternet Xətti (Biznes paketi, 200Mbps)",
            "trigger": "Biznes sərnişin profili — Fasiləsiz əlaqə tələb olunur",
            "biznes_deyeri": "Aztelekom: ARPU artımı"
        })

    # Çox baqaj → Smart çamadan izləyici
    if row.get("extra_baggage_tendency", 0) >= 2:
        recs.append({
            "company": "Aztelekom",
            "offer": "Ağıllı Çamadan İzləyici (IoT tracker) + internet paketi",
            "trigger": "Ortalama uçuşda 2+ əlavə baqaj",
            "biznes_deyeri": "Aztelekom: IoT cihaz satışı + data paketi gəliri"
        })

    # Tez-tez uçan → Smart Home kamera
    if row.get("flights_per_month", 0) >= 2:
        recs.append({
            "company": "Aztelekom",
            "offer": "Smart Home Təhlükəsizlik Kameraları Paketi",
            "trigger": "Ayda 2+ uçuş — evdə olmadığınız müddət üçün təhlükəsizlik",
            "biznes_deyeri": "Aztelekom: smart home satışı; AZAL: müştəri satisfaction"
        })

    # Məmnun müştəri → Lounge
    if row.get("satisfaction", 0) == 1 and row.get("loyalty_points", 0) > 10000:
        recs.append({
            "company": "AZAL",
            "offer": "AZAL VIP Lounge Giriş Kartı – loyallıq mükafatı",
            "trigger": "Yüksək məmnuniyyət + 10000+ loyallıq xalı",
            "biznes_deyeri": "AZAL: premium müştəri əlaqəsi"
        })

    # Gecikmiş uçuş kompensasiyası → Azİntelekom
    if row.get("delays", 0) > 1:
        recs.append({
            "company": "Azİntelekom",
            "offer": "Uçuş gecikmə kompensasiya sənədini Azİntelekom ilə anında imzala",
            "trigger": "Son dövrdə 2+ uçuş gecikməniz var",
            "biznes_deyeri": "Azİntelekom: rəqəmsal imza istifadəsi artımı"
        })

    # Ləğv edən müştəri → Azərpoçt
    if row.get("refund_requests", 0) > 0:
        recs.append({
            "company": "Azərpoçt",
            "offer": "Aeroportda qalan əşyaların ev ünvanına çatdırılması",
            "trigger": "Bilet ləğvetmə tarixçəniz var",
            "biznes_deyeri": "Azərpoçt: yeni kuryer sifarişi"
        })

    # Xarici uçan → Azərpoçt beynəlxalq poçt
    if row.get("intl_ratio", 0) > 0.5:
        recs.append({
            "company": "Azərpoçt",
            "offer": "Xaricdən alınan malları Azərbaycana Azərpoçt ilə göndər",
            "trigger": "Beynəlxalq uçuş nisbəti yüksəkdir",
            "biznes_deyeri": "Azərpoçt: beynəlxalq karqo gəliri"
        })

    # Ağır baqaj + AZAL ləğv → Azərpoçt evdən götürmə
    if row.get("extra_baggage_tendency", 0) >= 2 and row.get("next_booked_flight_days", 99) >= 0:
        recs.append({
            "company": "Azərpoçt",
            "offer": "Evdən hava limanına baqaj kuryer xidməti",
            "trigger": "Biletiniz var + ağır baqaj vərdişiniz var",
            "biznes_deyeri": "Azərpoçt: premium kuryer gəliri"
        })

    return recs


def azerpoct_recs(row):
    recs = []

    # Çox göndərən biznes → AZAL biznes-klass
    if row.get("parcels_sent", 0) > 7:
        recs.append({
            "company": "AZAL",
            "offer": "AZAL Biznes-Klass endirimli uçuş paketi",
            "trigger": "Aylıq 7+ bağlama göndərən biznes müştəri",
            "biznes_deyeri": "AZAL: yeni biznes müştəri"
        })

    # Beynəlxalq göndərmə istiqaməti → AZAL o istiqamətə endirim
    if row.get("intl_shipments", 0) > 1 and row.get("international_destination", "None") != "None":
        dest = row.get("international_destination", "")
        recs.append({
            "company": "AZAL",
            "offer": f"{dest} istiqamətinə AZAL geri dönüş uçuş endirimi",
            "trigger": f"Tez-tez {dest}-a beynəlxalq göndərməniz var",
            "biznes_deyeri": "AZAL: yeni marşrut müştərisi"
        })

    # Fast-Track → pickup müştəri
    if row.get("pickup_usage", 0) == 1:
        recs.append({
            "company": "AZAL",
            "offer": "AZAL Fast-Track hava limanı keçid xidməti",
            "trigger": "Azərpoçtda özü götürmə xidmətindən istifadə edirsiniz",
            "biznes_deyeri": "AZAL: premium add-on gəliri"
        })

    # İtirilmiş bağlama → Azİntelekom sığorta
    if row.get("lost_parcels", 0) > 0:
        recs.append({
            "company": "Azİntelekom",
            "offer": "Azİntelekom ilə Səyahət Sığortasını anında imzala",
            "trigger": "İtirilmiş bağlama təcrübəniz var — sığorta vacibdir",
            "biznes_deyeri": "Azİntelekom: rəqəmsal imza + sığorta gəliri"
        })

    # Qapıya çatdırma → AZAL VIP qarşılama
    if row.get("preferred_delivery_method", "") == "home":
        recs.append({
            "company": "AZAL",
            "offer": "AZAL VIP Qarşılama və Fast-Track Xidməti",
            "trigger": "Qapıya xidmət seçən 'komfort first' profili",
            "biznes_deyeri": "AZAL: premium segment artımı"
        })

    # E-ticarət satıcısı → Aztelekom Biznes internet
    if row.get("ecommerce_shipper", 0) == 1:
        recs.append({
            "company": "Aztelekom",
            "offer": "Aztelekom Biznes İnternet Paketi (Yüksək upload sürəti)",
            "trigger": "E-ticarət satıcısı profili — sürətli əlaqə tələb olunur",
            "biznes_deyeri": "Aztelekom: biznes tariflərə keçid"
        })

    # Daxili poçt (Naxçıvan) → ADY
    if row.get("international_destination", "") == "None" and row.get("parcels_sent", 0) > 3:
        recs.append({
            "company": "ADY",
            "offer": "Naxçıvan / daxili istiqamət üzrə ADY endirimli bilet",
            "trigger": "Daxili göndərmə fəallığı",
            "biznes_deyeri": "ADY: yeni müştəri"
        })

    # Kommunal ödəyən ənənəvi müştəri → Aztelekom
    if row.get("utility_payments_at_branch", 0) > 2:
        recs.append({
            "company": "Aztelekom",
            "offer": "Aztelekom Ev İnternet Paketi — kassada aktivləşdir",
            "trigger": "Filialda 2+ kommunal ödəniş — oflayn müştəri",
            "biznes_deyeri": "Aztelekom: yeni abunəçi"
        })

    # Pul köçürmə fəal → AZAL o ölkəyə uçuş
    if row.get("money_transfers_frequency", 0) > 1 and row.get("international_destination", "None") != "None":
        dest = row.get("international_destination", "")
        recs.append({
            "company": "AZAL",
            "offer": f"{dest}-a AZAL uçuş endirimi — pul köçürdüyünüz ölkəyə",
            "trigger": f"{dest}-a tez-tez pul köçürürsünüz",
            "biznes_deyeri": "AZAL: remittance corridor müştəriləri"
        })

    # Ləğv sifariş → AZAL-dan uçuş əvvəli bağlama yığımı
    if row.get("shipment_cancelled_lastmin", 0) > 0:
        recs.append({
            "company": "Azərpoçt",
            "offer": "Azərpoçt Kuryer: evdən hava limanına baqaj xidməti",
            "trigger": "Son anda ləğv edilmiş göndərmə qeydəniz var",
            "biznes_deyeri": "Azərpoçt: konversion artımı"
        })

    return recs


def aztelekom_recs(row):
    recs = []

    # Yüksək data istifadəçisi → Azcloud
    if row.get("high_data_user", 0) == 1 or row.get("average_monthly_usage_gb", 0) > 400:
        recs.append({
            "company": "Azcloud",
            "offer": "Azcloud 100GB Ehtiyat Yaddaş Paketi",
            "trigger": f"{row.get('average_monthly_usage_gb', 0):.0f} GB aylıq istifadə",
            "biznes_deyeri": "Aztelekom: cloud əlavə gəliri; Azcloud: yeni abunəçi"
        })

    # Gecikmiş ödəniş → Azİntelekom avtomatik ödəniş
    if row.get("payment_status", "") in ["unpaid", "overdue"] or row.get("payment_delay_days", 0) > 1:
        recs.append({
            "company": "Azİntelekom",
            "offer": "Azİntelekom ilə Avtomatik Ödəniş Aktivləşdirmə",
            "trigger": "Gecikmiş faktura qeydiniz var",
            "biznes_deyeri": "Aztelekom: ödəniş riski azalır; Azİntelekom: SSO istifadəsi artır"
        })

    # Postpaid müqavilə → Limitsiz premium
    if row.get("contract_type", 0) == 1:
        recs.append({
            "company": "Aztelekom",
            "offer": "Limitsiz Premium İnternet Sürət Paketi",
            "trigger": "Postpaid müqavilə müştərisi",
            "biznes_deyeri": "Aztelekom: ARPU artımı"
        })

    # Aşağı sürət → Fiber
    if row.get("speed_mbps", 100) <= 50:
        recs.append({
            "company": "Aztelekom",
            "offer": "Fiber Optik 100+ Mbps Yüksək Sürətli İnternet",
            "trigger": f"Hazırda {row.get('speed_mbps', '?')} Mbps — yüksəltmə fürsəti",
            "biznes_deyeri": "Aztelekom: upsell; müştəri: daha yaxşı təcrübə"
        })

    # Çox cihaz → Smart Home IoT
    if row.get("device_count", 0) >= 4:
        recs.append({
            "company": "Aztelekom",
            "offer": "Ağıllı Ev IoT Cihazları Paketi",
            "trigger": f"{row.get('device_count', 0)} aktiv cihaz qeydiyyatdadır",
            "biznes_deyeri": "Aztelekom: IoT ekosistemi genişlənməsi"
        })

    # Axşam aktiv istifadəçi → AZAL Flash Sale
    if row.get("peak_usage_time", "") in ["evening", "night"]:
        recs.append({
            "company": "AZAL",
            "offer": "AZAL Flash Sale gecə endirimləri — telefona bildiriş",
            "trigger": "Axşam/gecə saatlarda ən aktiv internet istifadəçisi",
            "biznes_deyeri": "AZAL: konversiya artımı; Aztelekom: data traffic"
        })

    # Hava limanına uzaq → AZAL transfer
    if row.get("installation_address_airport_km", 0) > 30:
        recs.append({
            "company": "AZAL",
            "offer": "AZAL bileti ilə xüsusi ev-aeroport transfer paketi",
            "trigger": f"Hava limanına {row.get('installation_address_airport_km', 0):.0f} km məsafə",
            "biznes_deyeri": "AZAL: add-on gəliri"
        })

    # Yüksək məmnuniyyət + uzun müddət → AZAL Lounge
    if row.get("customer_satisfaction_score", 0) >= 4 and row.get("tenure_months", 0) > 36:
        recs.append({
            "company": "AZAL",
            "offer": "AZAL VIP Lounge Giriş Hədiyyəsi — loyallıq mükafatı",
            "trigger": "Məmnuniyyət skoru 4+ × 3 ildən çox abunə",
            "biznes_deyeri": "Aztelekom: müştəri saxlama; AZAL: premium tanınma"
        })

    # GPON quraşdırma → Azİntelekom rəqəmsal müqavilə
    if row.get("service_type", "") == "GPON":
        recs.append({
            "company": "Azİntelekom",
            "offer": "GPON müqaviləni Azİntelekom ilə rəqəmsal imzala — ofisə getmə",
            "trigger": "GPON xidmət tipiniz var",
            "biznes_deyeri": "Azİntelekom: digital signature gəliri; Aztelekom: CX xərc azalır"
        })

    # Faktura bildirişini Azİntelekom ilə
    if row.get("payment_status", "") == "overdue":
        recs.append({
            "company": "Azİntelekom",
            "offer": "Azİntelekom üzərindən rəsmi ödəniş xatırlatması",
            "trigger": "Vaxtı keçmiş fakturanız var",
            "biznes_deyeri": "Azİntelekom: rəsmi bildirim kanalı; Aztelekom: ödəniş toplanması"
        })

    return recs


def ady_recs(row):
    recs = []

    # Tam rəqəmsal istifadəçi → AZAL endirimlər
    if row.get("app_ratio", 0) > 0.8:
        recs.append({
            "company": "AZAL",
            "offer": "AZAL Rəqəmsal Kanal Xüsusi Endirimlər (yalnız app)",
            "trigger": "Pullu əməliyyatların 80%+ rəqəmsal kanalda",
            "biznes_deyeri": "AZAL: rəqəmsal müştəri üçün daha aşağı xərc"
        })

    # Biznes class ADY → AZAL VIP Lounge
    if row.get("ticket_class", "") == "Business":
        recs.append({
            "company": "AZAL",
            "offer": "AZAL VIP Lounge + Sürətli Minik Xidməti",
            "trigger": "ADY-də Biznes Class istifadəçisisiniz",
            "biznes_deyeri": "AZAL: premium segment transfer"
        })

    # Yüksək loyallıq → Şirkətlərarası xal mübadiləsi
    if row.get("loyalty", 0) > 3000:
        recs.append({
            "company": "AZAL + ADY",
            "offer": "ADY loyallıq xallarını AZAL Miles-a çevir",
            "trigger": f"{row.get('loyalty', 0)} ADY xalınız var",
            "biznes_deyeri": "Hər iki şirkət: platform bağlılığı artır"
        })

    # Qrup səyahəti → endirim paketi
    if row.get("group_travel", 0) == 1:
        recs.append({
            "company": "ADY + AZAL",
            "offer": "Qrup Səyahət Endirim Paketi (6+ nəfər)",
            "trigger": "Qrup seyahəti vərdişiniz var",
            "biznes_deyeri": "ADY + AZAL: qrup satış gəliri"
        })

    # Naxçıvan marşrutu → AZAL daxili uçuş
    if row.get("route_type", "") == "nakhchivan":
        recs.append({
            "company": "AZAL",
            "offer": "Naxçıvan istiqamətinə AZAL daxili uçuş paketi",
            "trigger": "Naxçıvan istiqamətli ADY bilet tarixçəniz var",
            "biznes_deyeri": "AZAL: daxili marşrut dolulluq artımı"
        })

    # Ağır baqaj + Azərpoçt
    if row.get("heavy_luggage", 0) == 1:
        recs.append({
            "company": "Azərpoçt",
            "offer": "Azərpoçt Kuryer: stansiyadan ev ünvanına baqaj çatdırılması",
            "trigger": "Ağır baqaj profili",
            "biznes_deyeri": "Azərpoçt: stasionar müştəridən kuryer gəliri"
        })

    return recs


def azintelekom_recs(row):
    recs = []

    # Biometrik aktiv → AZAL Fast-Track
    if row.get("biometric_enabled", 0) == 1 or row.get("verified", 0) == 1:
        recs.append({
            "company": "AZAL",
            "offer": "Biometrik Azİntelekom ilə AZAL Sənədsiz Fast-Track Minik",
            "trigger": "Azİntelekom biometrik doğrulamanız aktivdir",
            "biznes_deyeri": "AZAL: mink prosesi sürətlənir; Azİntelekom: use-case artımı"
        })

    # Yüksək login → AZAL VIP rəqəmsal kampaniyalar
    if row.get("login_freq", 0) > 30:
        recs.append({
            "company": "AZAL",
            "offer": "AZAL VIP Rəqəmsal Kampaniyalar (in-app)",
            "trigger": f"Ayda {row.get('login_freq', 0):.0f} Azİntelekom girişi — çox aktiv",
            "biznes_deyeri": "AZAL: rəqəmsal konversiya; Azİntelekom: istifadə artımı"
        })

    # Çox xidmət → Telekom + Səyahət birləşmiş paket
    if row.get("services", 0) > 5:
        recs.append({
            "company": "Aztelekom + AZAL",
            "offer": "Aztelekom + AZAL Birləşmiş Abunə Paketi",
            "trigger": f"{row.get('services', 0)} Azİntelekom xidməti aktiv istifadəçi",
            "biznes_deyeri": "Hər iki şirkət: bundle gəliri + müştəri saxlama"
        })

    # Təhlükəsizlik xəbərdarlıqları → Azİntelekom sığorta
    if row.get("security_alerts", 0) > 1:
        recs.append({
            "company": "Azİntelekom",
            "offer": "Azİntelekom ilə Avtomatik Səyahət Sığortası İmzalama",
            "trigger": "Hesabınızda 2+ təhlükəsizlik xəbərdarlığı",
            "biznes_deyeri": "Azİntelekom: sığorta gəliri"
        })

    # Biznes sənəd tipi → Aztelekom Mini-ATS + AZAL Biznes
    if row.get("primary_document_types", "") == "Business":
        recs.append({
            "company": "Aztelekom",
            "offer": "Ofis üçün Mini-ATS + Virtual Nömrə Xidməti",
            "trigger": "Azİntelekom profilindəki birincil sənəd tipi: Biznes",
            "biznes_deyeri": "Aztelekom: biznes müştəri segmenti genişlənməsi"
        })
        recs.append({
            "company": "AZAL",
            "offer": "AZAL Korporativ Səyahət Paketi — birbaşa yönləndirilmə",
            "trigger": "Korporativ Azİntelekom profili",
            "biznes_deyeri": "AZAL: korporativ müqavilə gəliri"
        })

    # Yüksək rəqəmsal savad → Smart Home sensorlar
    if row.get("digital_literacy_score", "") == "High":
        recs.append({
            "company": "Aztelekom",
            "offer": "Aztelekom Ağıllı Ev Sensorları + Smart Hub",
            "trigger": "Yüksək rəqəmsal savad — IoT cihaz uyğunluğu",
            "biznes_deyeri": "Aztelekom: Smart Home ekosistemi satışı"
        })

    # Dövlət qulluqçusu → AZAL dövlət tarif
    if row.get("primary_document_types", "") == "Government" or row.get("govt_services_usage", 0) > 5:
        recs.append({
            "company": "AZAL",
            "offer": "AZAL Dövlət Sektoru Xüsusi Tariflər",
            "trigger": "Dövlət xidmətlərindən intensiv istifadə",
            "biznes_deyeri": "AZAL: dövlət seqmenti müştəri bazası"
        })

    # Daily backup → Aztelekom yüksək upload
    if row.get("data_sync_frequency", "") == "Daily":
        recs.append({
            "company": "Aztelekom",
            "offer": "Yüksək Upload Sürətli Aztelekom Tarifi (Azcloud sync üçün)",
            "trigger": "Gündəlik data sinxronizasiya istifadəçisi",
            "biznes_deyeri": "Aztelekom: premium tarif; Azcloud: istifadə artımı"
        })

    # Banking sənəd tipi → Aztelekom avto-ödəniş
    if row.get("primary_document_types", "") == "Banking":
        recs.append({
            "company": "Aztelekom",
            "offer": "Azİntelekom ilə Aztelekom Avto-Ödəniş Aktivləşdirmə",
            "trigger": "Maliyyə/kredit sənədi çox imzalayan profil",
            "biznes_deyeri": "Aztelekom: ödəniş riski azalır"
        })

    # Uşaq filtrini Azİntelekom ilə aktivləşdir
    if row.get("device_count", 0) >= 3 if "device_count" in row else False:
        recs.append({
            "company": "Aztelekom",
            "offer": "Azİntelekom ilə Uşaq İnternet Filtrini 1 Kliklə Aktivləşdir",
            "trigger": "Çox cihazlı ev — uşaq məzmun qoruması",
            "biznes_deyeri": "Aztelekom: müştəri arxayınlığı artımı"
        })

    return recs


def full_recommendation_engine(row):
    """Bütün şirkət rec funksiyalarını birləşdirir"""
    all_recs = []
    all_recs += azal_recs(row)
    all_recs += azerpoct_recs(row)
    all_recs += aztelekom_recs(row)
    all_recs += ady_recs(row)
    all_recs += azintelekom_recs(row)

    # Dedup by offer text
    seen = set()
    unique_recs = []
    for r in all_recs:
        key = r["offer"][:40]
        if key not in seen:
            seen.add(key)
            unique_recs.append(r)

    return unique_recs


# ─────────────────────────────────────────────
# 10. APPLY RECOMMENDATIONS
# ─────────────────────────────────────────────
print("\n" + "─" * 40)
print("10. Recommendation Engine tətbiq edilir...")
print("─" * 40)

feature_store["recommendations"] = feature_store.apply(full_recommendation_engine, axis=1)
feature_store["rec_count"]        = feature_store["recommendations"].apply(len)

# Flat format for CSV (one row per recommendation)
rec_rows = []
for _, row in feature_store.iterrows():
    for rec in row["recommendations"]:
        rec_rows.append({
            "phone_number": row["phone_number"],
            "segment": row["segment"],
            "churn_tier": row["churn_tier"],
            "target_company": rec["company"],
            "offer": rec["offer"],
            "trigger": rec["trigger"],
            "biznes_deyeri": rec["biznes_deyeri"],
        })

rec_df = pd.DataFrame(rec_rows)
rec_df.to_csv(f"{OUT}/recommendations.csv", index=False)

# ─────────────────────────────────────────────
# 11. FINAL CURATED FEATURES CSV
# ─────────────────────────────────────────────
curated.to_csv(f"{OUT}/curated_features.csv", index=False)
feature_store.drop(columns=["recommendations"]).to_csv(f"{OUT}/feature_store.csv", index=False)

# ─────────────────────────────────────────────
# 12. SUMMARY REPORT
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("AZCON PIPELINE — NƏTİCƏ XÜLASƏSİ")
print("=" * 60)

print(f"\n📦 Saxlanılan fayllar ({OUT}/):")
for f in sorted(os.listdir(OUT)):
    path = os.path.join(OUT, f)
    size = os.path.getsize(path) / 1024
    print(f"   {f:<40} {size:>8.1f} KB")

print(f"\n👥 Müştəri sayı       : {N:,}")
print(f"🔢 Feature Store sütunları: {feature_store.shape[1]}")
print(f"📋 Rec sayı (total)   : {len(rec_df):,}")
print(f"📊 Ortalama rec/müştəri: {feature_store['rec_count'].mean():.1f}")

print(f"\n🎯 Segment dağılımı:")
seg_counts = feature_store["segment"].value_counts()
for seg, cnt in seg_counts.items():
    print(f"   {seg:<25} {cnt:>5} müştəri ({cnt/N*100:.1f}%)")

print(f"\n📈 Churn risk dağılımı:")
churn_counts = feature_store["churn_tier"].value_counts()
for tier, cnt in churn_counts.items():
    print(f"   {tier:<25} {cnt:>5} müştəri ({cnt/N*100:.1f}%)")

print(f"\n🏢 Şirkət üzrə tövsiyə sayı:")
for company, cnt in rec_df["target_company"].value_counts().items():
    print(f"   {company:<30} {cnt:>5} tövsiyə")

print(f"\n✅ Pipeline uğurla tamamlandı!")

AzCon Feature Store Pipeline

✅ 5 şirkətin DB-si azcon_data/ qovluğuna saxlanıldı:
   aztelekom.csv — 2000 sətir × 24 sütun
   azal.csv — 2000 sətir × 20 sütun
   azerpoct.csv — 2000 sətir × 18 sütun
   ady.csv — 2000 sətir × 14 sütun
   azintelekom.csv — 2000 sətir × 14 sütun

✅ feature_store.csv saxlanıldı — 2000 sətir × 91 sütun
   Null dəyərlər: 0

✅ Curated features: 44 feature, 2000 müştəri

────────────────────────────────────────
7. Churn Prediction Model
────────────────────────────────────────
ROC-AUC : 0.7074
              precision    recall  f1-score   support

           0       0.79      0.99      0.88       312
           1       0.67      0.05      0.09        88

    accuracy                           0.79       400
   macro avg       0.73      0.52      0.48       400
weighted avg       0.76      0.79      0.70       400

────────────────────────────────────────
8. Customer Segmentation (KMeans)
────────────────────────────────────────

Segment distribution:
segment


In [3]:
import pandas as pd

# ─────────────────────────────────────────────
# 1. TEST CUSTOMERS (4 fərqli scenario)
# ─────────────────────────────────────────────

test_customers = pd.DataFrame([

    # 🔵 1. Convenience Traveler
    {
        "phone_number": "+994501234567",
        "travel_imminent": 1,
        "average_monthly_usage_gb": 420,
        "high_data_user": 1,
        "peak_usage_time": "evening",
        "installation_address_airport_km": 40,
        "travel_type": "Leisure",
        "flights_per_month": 2,
        "extra_baggage_tendency": 1,
        "intl_ratio": 0.6,
        "satisfaction": 1,
        "loyalty_points": 12000,
    },

    # 🔵 2. E-commerce Sender
    {
        "phone_number": "+994551112233",
        "parcels_sent": 15,
        "ecommerce_shipper": 1,
        "intl_shipments": 3,
        "international_destination": "Turkey",
        "pickup_usage": 1,
        "money_transfers_frequency": 2,
        "preferred_delivery_method": "home",
    },

    # 🔵 3. Business Traveler
    {
        "phone_number": "+994701234567",
        "travel_type": "Business",
        "business_ratio": 0.9,
        "flights_per_month": 6,
        "extra_baggage_tendency": 3,
        "loyalty_points": 25000,
        "satisfaction": 1,
        "delays": 2,
        "refund_requests": 1,
    },

    # 🔵 4. Digital Banking User
    {
        "phone_number": "+994771234567",
        "primary_document_types": "Banking",
        "login_freq": 55,
        "data_sync_frequency": "Daily",
        "security_alerts": 3,
        "device_count": 5,
        "verified": 1,
        "biometric_enabled": 1,
        "services": 8,
        "digital_literacy_score": "High",
    }

])

# ─────────────────────────────────────────────
# 2. MISSING COLUMNS → DEFAULT əlavə edirik
# ─────────────────────────────────────────────

# Engine-də istifadə olunan bütün sütunlar
required_cols = [
    "travel_imminent", "average_monthly_usage_gb", "high_data_user",
    "peak_usage_time", "installation_address_airport_km",
    "travel_type", "flights_per_month", "extra_baggage_tendency",
    "intl_ratio", "satisfaction", "loyalty_points",
    "parcels_sent", "ecommerce_shipper", "intl_shipments",
    "international_destination", "pickup_usage",
    "money_transfers_frequency", "preferred_delivery_method",
    "business_ratio", "delays", "refund_requests",
    "primary_document_types", "login_freq", "data_sync_frequency",
    "security_alerts", "device_count", "verified",
    "biometric_enabled", "services", "digital_literacy_score",
    "payment_status", "payment_delay_days", "contract_type",
    "speed_mbps", "device_count", "customer_satisfaction_score",
    "tenure_months", "service_type"
]

for col in required_cols:
    if col not in test_customers.columns:
        test_customers[col] = 0

# bəzi default stringlər
test_customers["payment_status"] = test_customers["payment_status"].replace(0, "paid")
test_customers["service_type"] = test_customers["service_type"].replace(0, "GPON")
test_customers["peak_usage_time"] = test_customers["peak_usage_time"].replace(0, "morning")
test_customers["preferred_delivery_method"] = test_customers["preferred_delivery_method"].replace(0, "branch")
test_customers["primary_document_types"] = test_customers["primary_document_types"].replace(0, "Personal")
test_customers["data_sync_frequency"] = test_customers["data_sync_frequency"].replace(0, "Weekly")
test_customers["digital_literacy_score"] = test_customers["digital_literacy_score"].replace(0, "Medium")

# ─────────────────────────────────────────────
# 3. APPLY ENGINE
# ─────────────────────────────────────────────

test_customers["recommendations"] = test_customers.apply(
    full_recommendation_engine,
    axis=1
)

# ─────────────────────────────────────────────
# 4. PRINT RESULTS
# ─────────────────────────────────────────────

for idx, row in test_customers.iterrows():
    print("\n" + "="*60)
    print(f"📱 Customer: {row['phone_number']}")
    print("="*60)

    recs = row["recommendations"]

    if not recs:
        print("❌ Heç bir recommendation çıxmadı")
    else:
        for i, rec in enumerate(recs, 1):
            print(f"\n{i}. 🏢 {rec['company']}")
            print(f"   🎁 Offer   : {rec['offer']}")
            print(f"   ⚡ Trigger : {rec['trigger']}")
            print(f"   💰 Biznes : {rec['biznes_deyeri']}")


📱 Customer: +994501234567

1. 🏢 Aztelekom
   🎁 Offer   : Ev interneti dondurma xidməti – Səyahət müddətinə pulsuz dondur
   ⚡ Trigger : AZAL biletiniz var, evdə internet boş qalmasın
   💰 Biznes : Aztelekom: loyalty artımı; AZAL: müştəri bağlılığı

2. 🏢 ADY/Taksi
   🎁 Offer   : Hava limanına xüsusi transfer xidməti
   ⚡ Trigger : Eviniz hava limanına 25+ km uzaqdır
   💰 Biznes : AZAL: add-on gəlir; ADY: yeni müştəri

3. 🏢 Aztelekom
   🎁 Offer   : Smart Home Təhlükəsizlik Kameraları Paketi
   ⚡ Trigger : Ayda 2+ uçuş — evdə olmadığınız müddət üçün təhlükəsizlik
   💰 Biznes : Aztelekom: smart home satışı; AZAL: müştəri satisfaction

4. 🏢 AZAL
   🎁 Offer   : AZAL VIP Lounge Giriş Kartı – loyallıq mükafatı
   ⚡ Trigger : Yüksək məmnuniyyət + 10000+ loyallıq xalı
   💰 Biznes : AZAL: premium müştəri əlaqəsi

5. 🏢 Azərpoçt
   🎁 Offer   : Xaricdən alınan malları Azərbaycana Azərpoçt ilə göndər
   ⚡ Trigger : Beynəlxalq uçuş nisbəti yüksəkdir
   💰 Biznes : Azərpoçt: beynəlxalq karqo gəliri

6.